# Analyse de Corrélation et Sélection des Caractéristiques (Feature Selection)

Ce notebook présente une étude approfondie des relations entre les budgets publicitaires (TV, Radio, Réseaux Sociaux) et les Ventes (`Sales`). L'objectif est d'identifier les variables les plus prédictives et de détecter d'éventuels problèmes de multicolinéarité pour optimiser notre futur modèle de régression.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.linear_model import LinearRegression
from sklearn.feature_selection import RFE

# Configuration des graphiques
%matplotlib inline
sns.set_theme(style="whitegrid")

## 1. Chargement des données nettoyées

In [ ]:
# Chargement du dataset nettoyé (sans valeurs manquantes ni valeurs aberrantes/outliers)
df = pd.read_csv('data/Clean_Data_HSS.csv')
print(f"Dimensions du jeu de données : {df.shape}")
df.head()

## 2. Analyse de Corrélation

Nous calculons la matrice de corrélation de Pearson pour mesurer la force et la direction de la relation linéaire entre nos variables numériques.

In [ ]:
# Calcul de la matrice de corrélation
corr_matrix = df.corr()
print("Matrice de Corrélation de Pearson :")
display(corr_matrix)

# Tracé de la carte de chaleur (Heatmap)
plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".4f", linewidths=0.5, vmin=-1, vmax=1)
plt.title("Matrice de Corrélation des Variables", fontsize=14, fontweight='bold')
plt.show()

### Observations sur la corrélation :
- **TV vs Sales (0.9995) :** Il existe une corrélation positive quasi-parfaite entre le budget TV et les ventes. Le canal TV est le principal moteur des ventes.
- **Radio vs Sales (0.8674) :** Forte corrélation positive.
- **Social Media vs Sales (0.5137) :** Corrélation positive modérée.
- **Multicolinéarité potentielle :** On note également de fortes corrélations entre les variables explicatives (ex: **TV et Radio ont une corrélation de 0.8672**). Cela suggère la présence de multicolinéarité, ce qui peut perturber l'interprétabilité des coefficients d'une régression linéaire standard.

## 3. Analyse de la Multicolinéarité (VIF - Variance Inflation Factor)

Le VIF permet de mesurer la gravité de la multicolinéarité. Un VIF > 5 ou 10 indique généralement une multicolinéarité problématique.

In [ ]:
# Sélection des variables indépendantes (features)
X = df[['TV', 'Radio', 'Social Media']]

# Calcul du VIF pour chaque feature
vif_data = pd.DataFrame()
vif_data["Feature"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(len(X.columns))]

print("Facteurs d'Inflation de la Variance (VIF) :")
display(vif_data)

### Analyse des VIF :
- **TV (VIF = 12.33)** et **Radio (VIF = 12.28)** ont des VIF élevés (> 10), confirmant une forte colinéarité. Ces deux canaux publicitaires tendent à évoluer de manière synchrone dans les dépenses marketing, rendant difficile la distinction de leur impact individuel s'ils sont utilisés ensemble sans précaution.
- **Social Media (VIF = 3.32)** a un VIF acceptable.

## 4. Sélection des Caractéristiques (Feature Selection)

Nous utilisons la méthode d'Élimination Récursive des Caractéristiques (RFE) basée sur un modèle de Régression Linéaire pour classer l'importance de nos variables.

In [ ]:
X = df[['TV', 'Radio', 'Social Media']]
y = df['Sales']

# Initialisation du modèle de régression
model = LinearRegression()

# Ajustement du modèle pour obtenir l'importance directe (Coefficients)
model.fit(X, y)

# Affichage des coefficients normés (ou coefficients bruts car les échelles sont comparables)
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_,
    'Abs_Coefficient': np.abs(model.coef_)
}).sort_values(by='Abs_Coefficient', ascending=False)

print("Importance des caractéristiques par Coefficients du modèle linéaire :")
display(coef_df)

# Application de RFE pour sélectionner la meilleure feature, puis les 2 meilleures
print("\n--- Sélection de Caractéristiques avec RFE ---")
for n in range(1, 4):
    rfe = RFE(estimator=model, n_features_to_select=n)
    rfe.fit(X, y)
    selected = X.columns[rfe.support_].tolist()
    print(f"Top {n} feature(s) sélectionnée(s) : {selected}")

## 5. Synthèse et Recommandations

### Conclusion de la Feature Selection :
1. **TV est la variable maîtresse :** Elle présente une corrélation de `0.9995` et le coefficient le plus fort dans la régression linéaire (`3.56`). Si l'on ne devait garder qu'une seule variable, ce serait **TV**.
2. **Le problème de multicolinéarité (TV et Radio) :** La forte corrélation linéaire entre TV et Radio entraîne des VIF élevés. Dans le modèle global, le coefficient de `Radio` est faible (`0.12`) par rapport à `TV` (`3.56`), car une grande partie de l'information de `Radio` est redondante avec celle de `TV`.
3. **Social Media :** Bien que corrélée à `0.51` avec les Ventes, son coefficient dans le modèle de régression linéaire est très proche de 0 (`0.003`). C'est la variable la moins utile dans une régression linéaire simple contenant déjà la TV.

### Recommandation pour la Modélisation :
- Si l'objectif est d'avoir un modèle simple et robuste sans risque d'instabilité des coefficients, un modèle contenant uniquement **TV** est extrêmement puissant ($R^2 \approx 0.999$).
- Si l'on souhaite intégrer les autres variables pour optimiser au maximum, il faudra surveiller la variance des coefficients ou utiliser des techniques de régularisation (Ridge/Lasso) ou des modèles basés sur les arbres de décision (Random Forest, XGBoost) qui gèrent mieux la multicolinéarité.